Preprocessing
Déduplication
Facteurs de risques (biblio)
Facteursde risques : Premières études d'impact

In [1]:
import pandas as pd
import numpy as np

# Visualization library
import altair as alt
alt.data_transformers.enable('default', max_rows=None)

# Dates management
import datetime

# For the computation of Kaplan-Meier estimates and log-rank tests
import lifelines

In [4]:
# Patients
df_person = pd.read_pickle("datap/df_person_fix.pkl")

# Visits
df_visit = pd.read_pickle("datap/df_visit.pkl")

# Diagnosis (condition)
df_cond = pd.read_pickle("datap/df_condition.pkl")

# Medication
df_bio_fix_final = pd.read_pickle("datap/df_bio_fix_final.pkl")

df_note = pd.read_pickle("datap/df_note.pkl")

In [5]:
df_visit_med = pd.merge(df_visit, df_bio_fix_final, on="visit_occurrence_id", how="left")

In [7]:
df_note.info()
df_note.head()

<class 'pandas.core.frame.DataFrame'>
Index: 7800 entries, 0 to 454
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   visit_occurrence_id  7800 non-null   int64         
 1   note_datetime        7800 non-null   datetime64[ns]
 2   note_id              7800 non-null   int64         
 3   cdm_source           7800 non-null   object        
 4   note_text            7800 non-null   object        
dtypes: datetime64[ns](1), int64(2), object(2)
memory usage: 365.6+ KB


,visit_occurrence_id,note_datetime,note_id,cdm_source,note_text
0,88969657,2025-02-07,86385800,EHR 1,Compte rendu de consultation\n\nPatient : [Nom...
1,82562657,2020-03-10,88729980,EHR 1,Compte rendu de consultation\n\nPatient : [Nom...
2,88795306,2020-01-27,88660889,EHR 1,Compte rendu de consultation\n\nPatient : [Nom...
3,83166022,2023-03-23,88805262,EHR 1,Compte rendu de consultation\n\nPatient : [Nom...
4,84710731,2022-04-08,85029453,EHR 1,Compte rendu de consultation\n\nPatient : [Nom...


In [12]:
df_visit.info()
df_visit.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1547 entries, 0 to 1546
Data columns (total 6 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   visit_occurrence_id   1547 non-null   float64       
 1   care_site_id          1547 non-null   object        
 2   visit_start_datetime  1547 non-null   datetime64[ns]
 3   visit_end_datetime    1546 non-null   datetime64[ns]
 4   visit_source_value    1547 non-null   object        
 5   person_id             1547 non-null   float64       
dtypes: datetime64[ns](2), float64(2), object(2)
memory usage: 72.6+ KB


,visit_occurrence_id,care_site_id,visit_start_datetime,visit_end_datetime,visit_source_value,person_id
0,88801206.0,Centre F.Sinoussi,2020-01-24,2020-01-26,Hospitalisés,82723807.0
1,81327360.0,Centre F.Sinoussi,2020-02-25,2020-03-05,Hospitalisés,80260379.0
2,89802376.0,GHU A.Fleming,2022-03-26,2022-03-27,Hospitalisés,85613365.0
3,81405613.0,Centre F.Sinoussi,2024-07-06,2024-07-07,Hospitalisés,80139607.0
4,80686859.0,Hopital M.Bres,2024-12-30,2025-01-08,Hospitalisés,87634325.0


In [8]:
df_visit_med.info()
df_visit_med.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6029 entries, 0 to 6028
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   visit_occurrence_id     6029 non-null   float64       
 1   care_site_id_x          6029 non-null   object        
 2   visit_start_datetime_x  6029 non-null   datetime64[ns]
 3   visit_end_datetime_x    6025 non-null   datetime64[ns]
 4   visit_source_value_x    6029 non-null   object        
 5   person_id_x             6029 non-null   float64       
 6   birth_datetime          5976 non-null   datetime64[ns]
 7   death_datetime          3276 non-null   datetime64[ns]
 8   gender_source_value     5976 non-null   object        
 9   cdm_source              5976 non-null   object        
 10  person_id_y             5976 non-null   float64       
 11  unique_person_id        5976 non-null   float64       
 12  care_site_id_y          5976 non-null   object  

,visit_occurrence_id,care_site_id_x,visit_start_datetime_x,visit_end_datetime_x,visit_source_value_x,person_id_x,birth_datetime,death_datetime,gender_source_value,cdm_source,...,unique_person_id,care_site_id_y,visit_start_datetime_y,visit_end_datetime_y,visit_source_value_y,measurement_id,measurement_datetime,concept_source_value,transformed_value,transformed_unit
0,88801206.0,Centre F.Sinoussi,2020-01-24,2020-01-26,Hospitalisés,82723807.0,1963-08-07,NaT,female,EHR 1,...,82723807.0,Centre F.Sinoussi,2020-01-24,2020-01-26,Hospitalisés,81228660.0,2020-01-24,crp,4.53,mg/L
1,88801206.0,Centre F.Sinoussi,2020-01-24,2020-01-26,Hospitalisés,82723807.0,1963-08-07,NaT,female,EHR 1,...,82723807.0,Centre F.Sinoussi,2020-01-24,2020-01-26,Hospitalisés,89065276.0,2020-01-24,urea,3.71,mmol/L
2,88801206.0,Centre F.Sinoussi,2020-01-24,2020-01-26,Hospitalisés,82723807.0,1963-08-07,NaT,female,EHR 1,...,82723807.0,Centre F.Sinoussi,2020-01-24,2020-01-26,Hospitalisés,85894139.0,2020-01-24,bmi,27.84,kg.cm^-2
3,88801206.0,Centre F.Sinoussi,2020-01-24,2020-01-26,Hospitalisés,82723807.0,1963-08-07,NaT,female,EHR 1,...,82723807.0,Centre F.Sinoussi,2020-01-24,2020-01-26,Hospitalisés,85641747.0,2020-01-24,hb,11.04,g/dL
4,81327360.0,Centre F.Sinoussi,2020-02-25,2020-03-05,Hospitalisés,80260379.0,1965-09-29,NaT,male,EHR 1,...,80260379.0,Centre F.Sinoussi,2020-02-25,2020-03-05,Hospitalisés,85453397.0,2020-02-25,hb,15.00,g/dL


In [9]:
print(f"We have {df_visit_med.visit_occurrence_id.nunique()} unique patient ids in this dataset.")

We have 1547 unique patient ids in this dataset.


In [11]:
df_value_count = df_visit_med["visit_occurrence_id"].value_counts()
n_numerous = len(df_value_count[df_value_count > 1])
print(f"{n_numerous} patients have more than one visit")

1494 patients have more than one visit


In [13]:
df_value_count = df_visit["person_id"].value_counts()
n_numerous = len(df_value_count[df_value_count > 1])
print(f"{n_numerous} patients have more than one visit")

0 patients have more than one visit
